# Model Training

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from pathlib import Path
import sys

root = Path.cwd().resolve()
for parent in [root] + list(root.parents):
    if (parent / "ml_engine").exists():
        sys.path.insert(0, str(parent))
        break
else:
    sys.path.insert(0, str(root))

from ml_engine.train import train_logistic_regression, train_random_forest, train_xgboost, save_model, save_scaler, save_feature_names
from ml_engine.feature_engineering import prepare_training_data

In [2]:
final_df = pd.read_csv('provider_master.csv')

In [3]:
CORE_FEATURES = [
    "TotalClaims", "TotalReimbursed", "AvgReimbursed",
    "AvgClaimDuration", "AvgDaysInHospital",
    "UniquePatients", "UniqueAttendPhys",
    "SameAttendOperRate", "AvgChronicConds", "InpatientRatio",
]
X, y = prepare_training_data(final_df, CORE_FEATURES, label_column='FraudLabel')

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train-Test Split Results:")
print(f"  X_train : {X_train.shape}  -  {len(X_train):,} providers for training")
print(f"  X_test  : {X_test.shape}   -  {len(X_test):,}  providers for testing")
print()
print("Fraud ratio check (stratify working correctly):")
print(f"  Original  : {y.mean()*100:.1f}% fraud")
print(f"  Train set : {y_train.mean()*100:.1f}% fraud  - should match original")
print(f"  Test  set : {y_test.mean()*100:.1f}% fraud  - should match original")

Train-Test Split Results:
  X_train : (4328, 10)  -  4,328 providers for training
  X_test  : (1082, 10)   -  1,082  providers for testing

Fraud ratio check (stratify working correctly):
  Original  : 9.4% fraud
  Train set : 9.4% fraud  - should match original
  Test  set : 9.3% fraud  - should match original


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=CORE_FEATURES, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=CORE_FEATURES, index=X_test.index)

print("Scaling Results:")
print("\nBefore scaling (X_train stats):")
print(X_train[["TotalReimbursed", "InpatientRatio", "TotalClaims"]].describe().round(2))
print("\nAfter scaling (X_train_scaled stats):")
print(X_train_scaled[["TotalReimbursed", "InpatientRatio", "TotalClaims"]].describe().round(2))

Scaling Results:

Before scaling (X_train stats):
       TotalReimbursed  InpatientRatio  TotalClaims
count          4328.00         4328.00      4328.00
mean          99555.50            0.15        98.07
std          259764.97            0.29       259.27
min               0.00            0.00         1.00
25%            4477.50            0.00        10.00
50%           19480.00            0.00        31.00
75%           85002.50            0.13        85.00
max         5996050.00            1.00      8240.00

After scaling (X_train_scaled stats):
       TotalReimbursed  InpatientRatio  TotalClaims
count          4328.00         4328.00      4328.00
mean             -0.00            0.00        -0.00
std               1.00            1.00         1.00
min              -0.38           -0.50        -0.37
25%              -0.37           -0.50        -0.34
50%              -0.31           -0.50        -0.26
75%              -0.06           -0.04        -0.05
max              22.70     

In [6]:
print("Before SMOTE:")
print(f"  Training set size : {len(X_train_scaled):,}")
print(f"  Not Fraud         : {(y_train == 0).sum():,} ({(y_train == 0).mean()*100:.1f}%)")
print(f"  Fraud             : {(y_train == 1).sum():,} ({(y_train == 1).mean()*100:.1f}%)")

Before SMOTE:
  Training set size : 4,328
  Not Fraud         : 3,923 (90.6%)
  Fraud             : 405 (9.4%)


In [7]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print("\nAfter SMOTE:")
print(f"  Training set size : {len(X_train_resampled):,}")
print(f"  Not Fraud         : {(y_train_resampled == 0).sum():,} ({(y_train_resampled == 0).mean()*100:.1f}%)")
print(f"  Fraud             : {(y_train_resampled == 1).sum():,} ({(y_train_resampled == 1).mean()*100:.1f}%)")
print()


After SMOTE:
  Training set size : 7,846
  Not Fraud         : 3,923 (50.0%)
  Fraud             : 3,923 (50.0%)



# Train Baseline Model

In [8]:
lr_baseline = train_logistic_regression(X_train_resampled, y_train_resampled, random_state=42)
print("Baseline Logistic Regression trained")
print(f"  Trained on : {len(X_train_resampled):,} samples (after SMOTE)")
print(f"  Features   : {len(CORE_FEATURES)}")

Baseline Logistic Regression trained
  Trained on : 7,846 samples (after SMOTE)
  Features   : 10


In [9]:
rf_model = train_random_forest(X_train_resampled, y_train_resampled, random_state=42)
xgb_model = train_xgboost(X_train_resampled, y_train_resampled, random_state=42)
print("Random Forest and XGBoost models trained")

c:\Users\yuvar\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:12:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest and XGBoost models trained


In [11]:
save_model(lr_baseline, '../models/logistic_regression.pkl')
save_model(rf_model, '../models/random_forest.pkl')
save_model(xgb_model, '../models/xgboost.pkl')
save_scaler(scaler, '../models/scaler.pkl')
save_feature_names(CORE_FEATURES, '../models/feature_names.pkl')
print("Saved models/logistic_regression.pkl, models/random_forest.pkl, models/xgboost.pkl, models/scaler.pkl, models/feature_names.pkl")

Saved models/logistic_regression.pkl, models/random_forest.pkl, models/xgboost.pkl, models/scaler.pkl, models/feature_names.pkl
